# Day II - A) Direct vs iterative solvers

- **<font color='green'>[RUN & OBSERVE]</font>** $\rightarrow$ the cell should be run directly without modification
- **<font color='orange'>[RUN & PLAY]</font>** $\rightarrow$ the cell can be run directly, but some parameters should be changed interactively
- **<font color='red'>[FILL & RUN]</font>**    $\rightarrow$ the cell should be filled before being run
- **<font color='magenta'>[FILL & PLAY]</font>** $\rightarrow$ the cell should be filled, and then some parameters should be changed interactively.


---
## Outlines:

<a id='outlines'></a>

- [**1) Problem definition**](#1_problem_definition)
- [**2) 2D analysis**](#2_2d_analysis)
    - [a) Geometry and mesh](#2a_geometry_and_mesh)
    - [b) Weak form discretization](#2b_weak_form_discretization)
    - [c) Direct vs iterative solver](#2c_solver_comparison)
- [**3) 3D analysis**](#3_3d_analysis)
    - [a) Geometry and mesh](#3a_geometry_and_mesh)
    - [b) Weak form discretization](#3b_weak_form_discretization)
    - [c) Direct vs iterative solver](#3c_solver_comparison)
- [**4) Comparison: the role of bandwidth**](#4_comparison)
- [**5) Conclusion**](#5_conclusion)

|**<u>Objective</u>** : Compare direct and iterative solvers on 2D and 3D Poisson problems, and relate solver performance to matrix structure|
|---|
|*Indicative duration* : 45 min|


---

<a id='1_problem_definition'></a>

## 1) Problem definition

In the previous session, we assembled and solved a small linear system with a direct solver. Here we ask a natural question: **as the problem grows to 2D and 3D, is a direct solver still the best choice?** We will compare two families of solvers — *direct* (factorization-based) and *iterative* (Krylov subspace) — on the same Poisson problem.

We consider the **Poisson equation**, which models several physical phenomena (electrostatics, membrane problems, steady heat conduction, 2D isotropic and source-free magnetostatics, *etc.*), and reads:

$$ -\text{div}\, (h \, \nabla a) = s, $$

on the following domain $\Omega$:

![image-3.png](attachment:image-3.png)

The physical state $a$ is set to $0$ on the external boundary $\partial\Omega$ (homogeneous Dirichlet condition), the diffusion coefficient $h=1$ is constant on the whole domain, and the source term is $s=1$ inside `Int` and $s=0$ in `Ext`.

The weak form is obtained by multiplying the strong form by a test function $v$ and integrating over $\Omega$. Integrating by parts (Green's identity) gives:

$$\forall v \in H^1_0(\Omega), \quad
-\int_{\Gamma} v\,(h\,\nabla a) \cdot \underline{n} \,ds + \int_\Omega \nabla v \cdot (h\,\nabla a) \,dx = \int_\Omega v\,s\,dx.$$

Since $v = 0$ on $\partial\Omega$ (homogeneous Dirichlet condition), the boundary integral vanishes, and the **weak form** reads:

$$\boxed{\forall v \in H^1_0(\Omega), \quad
\underbrace{\int _\Omega \nabla v \cdot (h\,\nabla a)\,dx}_{\text{Bilinear form } a(v,a)} = \underbrace{\int _{\Omega }v\,s\,dx}_{\text{Linear form } b(v)}}$$

where $H^1_0(\Omega)$ is the Sobolev space of functions with square-integrable gradients that vanish on the boundary:

$$ H^1_0(\Omega) = \left\{ v \in L^2(\Omega) \;\middle|\; \int_\Omega \bigl(|v|^2 + |\nabla v|^2\bigr)\,dx < \infty, \;\; v = 0 \text{ on } \partial\Omega \right\}.$$


---

<a id='2_2d_analysis'></a>

## 2) 2D analysis

### 2.a) Geometry and mesh

<a id='2a_geometry_and_mesh'></a>

We first solve the Poisson equation on a 2D domain. The geometry is built from two subdomains using CSG (Constructive Solid Geometry):
- the **inner disk** `Int` (radius 0.1, where the source is active)
- the **outer annulus** `Ext` (radius 1, where the source vanishes)
- the **outer boundary** `dOmega` (where $a=0$)

The mesh is generated with `maxh=0.01178`, which gives a few thousand elements.

|**<font color='green'>[RUN & OBSERVE]</font>**|
|---|

In [ ]:
from netgen.geom2d import CSG2d, Circle
from ngsolve import Mesh
from ngsolve.webgui import Draw

#-------------------------------------------------------------
# 1) Create the solids (inner disk and outer annulus)
Int = Circle( center=(0,0), radius=0.1, mat = "Int" )
Out = Circle( center=(0,0), radius=1, mat = "Ext", bc = "dOmega" )
Ext = Out - Int

# 2) Build the CSG geometry
Omega2D = CSG2d()
Omega2D.Add(Int)
Omega2D.Add(Ext)

#-------------------------------------------------------------
# 3) Generate the mesh and plot
mesh2D = Mesh(Omega2D.GenerateMesh(maxh=0.01178))
print(f"Number of elements: {mesh2D.ne}, Number of nodes: {mesh2D.nv}")
Draw(mesh2D)

### 2.b) Weak form discretization

<a id='2b_weak_form_discretization'></a>

We now define the discrete weak form. This requires:
- a **finite element space** `fes2D` (continuous $P_1$ elements with homogeneous Dirichlet condition on `dOmega`);
- a **bilinear form** `K2D`: $\int_\Omega \nabla v \cdot \nabla a \, dx$ (LHS of the weak form, involves both test and trial functions);
- a **linear form** `s2D`: $\int_\Omega v \cdot s \, dx$ restricted to `Int` (RHS, involves only the test function).

|**<font color='red'>[FILL & RUN]</font>**|
|---|

In [ ]:
from ngsolve import H1, grad, dx, BilinearForm, LinearForm

# Finite element space: continuous P1, homogeneous Dirichlet on "dOmega"
fes2D = H1(mesh2D, dirichlet="dOmega")

# Trial function (unknown) and test function
a     = fes2D.TrialFunction()
aStar = fes2D.TestFunction()

# Bilinear form (LHS): a(v, a) = ∫Ω ∇v · ∇a dx
K2D = BilinearForm( grad(aStar) * grad(a) * dx )

# Linear form (RHS): b(v) = ∫Int v · s dx, with s = 1
s2D = LinearForm( aStar * 1 * dx("Int") )

### 2.c) Direct vs iterative solver

<a id='2c_solver_comparison'></a>

Once the FES, bilinear and linear forms are defined, we assemble and solve the linear system.

> *Note: although the code writes `K2D.mat.Inverse(...)`, NGSolve does **not** compute the explicit matrix inverse. It performs a sparse Cholesky factorization, which is the standard and efficient approach for symmetric positive-definite systems.*

We use two families of solvers:
- **Direct solver** (sparse Cholesky): factorizes the matrix to solve the system *exactly* (up to round-off). The factorization is computationally expensive — $O(b\,N^2)$ in flops for a sparse matrix of size $N$ and bandwidth $b$ — but the factorization can be **reused** to solve for different right-hand sides at a cost of only $O(b\,N)$ each. Direct solvers are nearly **insensitive to the condition number** $\kappa(K)$, but their cost grows with the **bandwidth** $b$.
- **Iterative solver** (Conjugate Gradient, CG): builds a sequence of approximations $x_k$ that converges to the exact solution; the iteration stops when the residual falls below a given tolerance. Each iteration costs $O(n_{\text{nz}})$ (one sparse matrix–vector product plus a few vector operations). The convergence rate depends on the **condition number**: the number of iterations scales like $O(\sqrt{\kappa(K)})$. Iterative solvers are **insensitive to bandwidth**.

|1. Compare the computation times of the direct and iterative solvers on the 2D mesh.
|2. How many times faster is each?
|3. Which solver would you choose for a 2D problem, and why?|
|---|

|**<font color='green'>[RUN & OBSERVE]</font>**|
|---|

In [ ]:
from ngsolve.krylovspace import CGSolver
from ngsolve import GridFunction
import time

###########################
# Assembly of the matrix and right-hand side (RHS)
K2D.Assemble()   # matrix
s2D.Assemble()   # RHS vector

###########################
# Direct solver (sparse Cholesky)
solutionDirect = GridFunction(fes2D)
tic = time.time()
directSolver = K2D.mat.Inverse(freedofs=fes2D.FreeDofs(), inverse="sparsecholesky")
solutionDirect.vec.data = directSolver * s2D.vec
t_direct_2D = time.time() - tic
print(f"Time direct solver (2D): {t_direct_2D*1000:.0f} ms")

###########################
# Iterative solver (Conjugate Gradient)
solutionIterative = GridFunction(fes2D)
tic = time.time()
iterativeSolver = CGSolver(K2D.mat, freedofs=fes2D.FreeDofs(), atol=1e-12, maxiter=1000)
solutionIterative.vec.data = iterativeSolver * s2D.vec
t_iter_2D = time.time() - tic
print(f"Time iterative solver (2D): {t_iter_2D*1000:.0f} ms")

# Visualize the gradient of the direct solution
print("\nPlot of ∇(solution) — 2D")
Draw(grad(solutionDirect), mesh2D,
     vectors={"grid_size": 20},
     settings={"Objects": {"Wireframe": False}})

On this 2D mesh, the direct solver is competitive or faster because the bandwidth is small (a few hundred), so the $O(b\,N^2)$ cost is still manageable. But what happens in 3D, where the bandwidth grows with mesh size? Let's find out.

---

<a id='3_3d_analysis'></a>

## 3) 3D analysis

### 3.a) Geometry and mesh

<a id='3a_geometry_and_mesh'></a>

We now solve the same Poisson problem on the **3D** domain: an inner sphere (radius 0.1, source active) inside an outer sphere (radius 1, source zero), with $a=0$ on the outer boundary. We choose `maxh=0.051` to keep approximately the same number of degrees of freedom as the 2D case (be patient, 3D mesh generation may take a few seconds).

|**<font color='green'>[RUN & OBSERVE]</font>**|
|---|

In [ ]:
from netgen.csg import CSGeometry, Sphere, Pnt

#---------------------------------------------------------------
# Construction of the 3D geometry
Omega3D = CSGeometry()
Int = Sphere(Pnt(0,0,0), 0.1).mat("Int")
Out = Sphere(Pnt(0,0,0), 1).bc("dOmega")
Ext = Out - Int

Omega3D.Add(Int)
Omega3D.Add(Ext)

#---------------------------------------------------------------
# Meshing
ngmesh3D = Omega3D.GenerateMesh(maxh=0.051)
mesh3D = Mesh(ngmesh3D)

print(f"Number of elements: {mesh3D.ne}, Number of nodes: {mesh3D.nv}")
Draw(mesh3D, clipping={"y": 1, "z": 0, "dist": 0})

### 3.b) Weak form discretization

<a id='3b_weak_form_discretization'></a>

Define the FES, bilinear and linear forms for the 3D problem, then assemble the matrix `K3D` and the RHS `s3D`.

|**<font color='red'>[FILL & RUN]</font>**|
|---|

In [ ]:
#---------------------------------------------------------------
# Finite element space
fes3D = H1(mesh3D, dirichlet="dOmega")

#---------------------------------------------------------------
# Integral equations
aStar = fes3D.TestFunction()
a     = fes3D.TrialFunction()
K3D   = BilinearForm( grad(aStar) * grad(a) * dx )
s3D   = LinearForm( aStar * 1 * dx("Int") )

#---------------------------------------------------------------
# Assembly
K3D.Assemble()
s3D.Assemble()

### 3.c) Direct vs iterative solver

<a id='3c_solver_comparison'></a>

> *Warning: the tolerance `atol=1e-12` is very strict for an unpreconditioned CG. If the solver reaches `maxiter=1000`, the displayed time is a **lower bound** on the true cost. Try relaxing it to `atol=1e-8`, or adding a preconditioner (e.g. multigrid), to see the difference.*

|1. Compare the computation times of the direct and iterative solvers on the 3D mesh.
|2. How does the comparison differ from the 2D case?
|3. Which solver would you choose for a 3D problem, and why?|
|---|

|**<font color='green'>[RUN & OBSERVE]</font>**|
|---|

In [ ]:
# Direct solver (sparse Cholesky)
solutionDirect3D = GridFunction(fes3D)
tic = time.time()
directSolver3D = K3D.mat.Inverse(freedofs=fes3D.FreeDofs(), inverse="sparsecholesky")
solutionDirect3D.vec.data = directSolver3D * s3D.vec
t_direct_3D = time.time() - tic
print(f"Time direct solver (3D): {t_direct_3D*1000:.0f} ms")

# Iterative solver (Conjugate Gradient)
solutionIterative3D = GridFunction(fes3D)
tic = time.time()
iterativeSolver3D = CGSolver(K3D.mat, freedofs=fes3D.FreeDofs(), atol=1e-12, maxiter=1000)
solutionIterative3D.vec.data = iterativeSolver3D * s3D.vec
t_iter_3D = time.time() - tic
print(f"Time iterative solver (3D): {t_iter_3D*1000:.0f} ms")

# Visualize the gradient of the iterative solution
print("\nPlot of ∇(solution) — 3D")
Draw(grad(solutionIterative3D), mesh3D,
     vectors={"grid_size": 20},
     clipping={"y": 1, "z": 0, "dist": 0},
     settings={"Objects": {"Wireframe": False}})

This time, the iterative solver is clearly faster! The reason is not the number of degrees of freedom (which is similar to the 2D case), but the **structure** of the matrix.

---

<a id='4_comparison'></a>

## 4) Comparison: the role of bandwidth

The key quantity is the **bandwidth** $b$ of the sparse matrix: the maximum distance between the diagonal and a non-zero entry. For the FE Laplacian, the non-zero entries in row $i$ correspond to the **neighboring nodes** of node $i$. The more neighbors, the larger the bandwidth, and the higher the cost of a direct factorization ($O(b\,N^2)$).

To make the comparison visually clear, we reorder the matrices with the **Cuthill-McKee algorithm**, a graph-based permutation that minimizes the bandwidth and concentrates the non-zeros around the diagonal.

|Look at the two sparsity patterns below. What do you observe? How does the "half-bandwidth" (distance from the diagonal to the furthest non-zero) compare between 2D and 3D?|
|---|

***Hint:*** in 2D a node has on average ~6 neighboring elements; in 3D up to ~30.

![image-4.png](attachment:image-4.png)

|**<font color='green'>[RUN & OBSERVE]</font>**|
|---|

In [ ]:
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import reverse_cuthill_mckee
import matplotlib.pyplot as plt

fig, axs = plt.subplots(1, 2, figsize=(12, 5))

# --- 2D matrix: apply Cuthill-McKee reordering to minimize bandwidth ---
rows, cols, vals = K2D.mat.COO()
K2Dsp = csr_matrix((vals, (rows, cols)))
perm2D = reverse_cuthill_mckee(K2Dsp)
K2D_perm = K2Dsp[perm2D, :][:, perm2D]
axs[0].spy(K2D_perm, markersize=1)
axs[0].set_title("FE matrix 2D\n(Cuthill-McKee reordered)")
axs[0].set_xlabel("Column")
axs[0].set_ylabel("Row")

# --- 3D matrix: apply Cuthill-McKee reordering to minimize bandwidth ---
rows, cols, vals = K3D.mat.COO()
K3Dsp = csr_matrix((vals, (rows, cols)))
perm3D = reverse_cuthill_mckee(K3Dsp)
K3D_perm = K3Dsp[perm3D, :][:, perm3D]
axs[1].spy(K3D_perm, markersize=1)
axs[1].set_title("FE matrix 3D\n(Cuthill-McKee reordered)")
axs[1].set_xlabel("Column")
axs[1].set_ylabel("Row")

plt.tight_layout()
plt.show()

---

<a id='5_conclusion'></a>

## 5) Conclusion

The time complexity and memory footprint of **direct solvers** depend on the **bandwidth** $b$ of the matrix (cost $O(b\,N^2)$, fill-in memory $O(b\,N)$), while **iterative solvers** are insensitive to bandwidth. We therefore conclude:

|**1D / 2D problems** → low connectivity (~6 neighbors per node) → small bandwidth → **direct solvers** are often preferred|
|---|
|**3D problems** → high connectivity (up to ~30 neighbors per node) → large bandwidth → **iterative solvers** scale much better|

However, iterative solvers are sensitive to the condition number $\kappa(K)$: the CG iteration count scales as $O(\sqrt{\kappa(K)})$. For practical 3D problems, a good **preconditioner** (e.g. algebraic multigrid) is almost always needed to keep the iteration count reasonable.

---

# Take home message

**The choice of solver depends on the matrix structure (bandwidth / sparsity pattern), not just the matrix size.**

- **Direct solvers** (Cholesky, LU): robust, exact, reusable for multiple RHS, but cost and memory grow with bandwidth. Ideal for 1D/2D.
- **Iterative solvers** (CG, GMRES, etc.): memory-efficient, insensitive to bandwidth, but convergence depends on the condition number. Ideal for 3D (with a good preconditioner).

Neither is universally superior — the right tool depends on the problem.

---

## To go further

- [Sparse matrices and direct methods (French course notes, IRISA)](https://www.irisa.fr/sage/jocelyne/cours/INSA/chapcreux2014.pdf)
- [The conjugate gradient method, quickly and painlessly (R. B. Smith, 1994)](http://www.cs.cmu.edu/~quake-papers/painless-conjugate-gradient.pdf)
- [Lecture: Direct and iterative solvers (UIUC CS 357)](https://courses.engr.illinois.edu/cs357/su2014/lectures/lecture08.pdf)
- [Sparse linear systems — course notes (E. Darrigrand-Lacarrée, Univ. Rennes)](https://perso.univ-rennes1.fr/eric.darrigrand-lacarrieu/Teaching/PdfFiles/PolyF04cours.pdf)
- [Méthodes directes (F. Rémoué, Univ. Brest)](http://web.univ-ubs.fr/lmba/frenod/IMG/DocEtudiant/MTH1504/methodes-directes.pdf)

---

[*Back to top - outlines*](#outlines)